# 02 · Cruzar DOS bases (record linkage A ↔ B) — nivel INTERMEDIO

**Que hace**: encuentra que empresas de la **Base A** estan tambien en la **Base B**
aunque el NIT venga distinto o el nombre este escrito diferente. Entrega la **carpeta
del estandar de salida** (`golden.parquet`, `correlativa.parquet`, `excel/`,
`manifest.json`) y una vista `PARES_CRUZADOS` con las empresas halladas en ambas.

**Configuracion**: cada base se describe con un `EspecBase` (archivo, hoja y el
nombre que SUS columnas tienen). Puede declarar una base como **confiable**
(`trusted`): sus registros no se deduplican entre si, solo se cruzan contra la otra.

**Pasos**: Celda A (entorno) → Celda A1 (rutas) → Celda A2 (definiciones) →
**Celda B** (editar y correr).

> En Colab Free, suba las bases a `/content/datos` y descargue `/content/resultados`
> antes de cerrar la sesion; el disco local es efimero.
>
> Motor: `rues-linker 0.22.4` (`link`, sobre el Orchestrator de produccion; el
> resultado es un `ResultadoLinkage` con el contrato de salida 1.0).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.22.4"
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"


def _version_ok(v: str, objetivo: str = VERSION_OBJETIVO) -> bool:
    """True si la version instalada coincide con el release objetivo."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) == _t(objetivo)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza exactamente rues-linker 0.22.4 desde el tag oficial."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada):
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub...")
    else:
        print(f"rues-linker {instalada} no coincide con {VERSION_OBJETIVO}; instalando...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada. "
            "Reinicie el entorno y vuelva a ejecutar esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_OBJETIVO}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS  (donde estan sus datos y donde quedan los resultados)
# ══════════════════════════════════════════════════════════════════════════
# En Colab Free suba sus archivos a /content/datos (los notebooks 01-04 no
# montan Drive) y descargue RUTA_RESULTADOS antes de cerrar la sesion: el
# disco local es efimero. Los nombres de archivo se declaran en la Celda B.
# tests/test_notebooks_contrato.py sobreescribe estas rutas para correr el
# notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/datos")  # carpeta con sus archivos
RUTA_RESULTADOS = Path("/content/resultados/02")  # carpeta de salida

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════

# ── Lectura y validacion de bases (fail-fast con mensajes accionables) ──────

import gc
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from record_linkage import escribir_resultado, exportar_vistas, leer_resultado, link


def _norm(texto: str) -> str:
    """Normaliza un nombre de columna: sin tildes, mayusculas, sin espacios extremos."""
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).strip().upper()


def leer_base(
    ruta: str | Path,
    hoja: str | int = 0,
    fila_encabezado: int = 0,
    columnas_texto: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Lee CSV/XLSX validando existencia y devolviendo columnas normalizadas.

    Args:
        ruta: ruta local al archivo .csv, .xlsx o .xls.
        hoja: nombre o indice de hoja (solo Excel).
        fila_encabezado: fila (0-index) donde estan los encabezados.
        columnas_texto: columnas a forzar como texto (p. ej. NIT, para no
            perder ceros a la izquierda ni convertir a notacion cientifica).

    Returns:
        DataFrame con encabezados normalizados (sin tildes, en mayusculas).

    Raises:
        FileNotFoundError: si la ruta no existe (con la carpeta que si existe).
        ValueError: si la extension no es soportada.
    """
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:12]) or "(vacia)"
        raise FileNotFoundError(
            f"No existe: {p}\n  La carpeta mas cercana que SI existe es: {padre}\n"
            f"  Contiene: {vistos}\n  Corrija la ruta o el nombre del archivo (copielo exacto)."
        )
    dtype = {c: "string" for c in columnas_texto}
    ext = p.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(p, header=fila_encabezado, dtype=dtype, encoding="utf-8")
    elif ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(p, sheet_name=hoja, header=fila_encabezado, dtype=dtype)
    else:
        raise ValueError(f"Extension no soportada: '{ext}'. Use .csv o .xlsx")
    df.columns = [_norm(c) for c in df.columns]
    return df


def exigir_columnas(df: pd.DataFrame, requeridas: dict[str, str], contexto: str) -> pd.DataFrame:
    """Renombra columnas del usuario al contrato NIT/RAZON_SOCIAL/... y valida.

    Args:
        df: base leida.
        requeridas: {nombre_en_su_archivo: nombre_contrato}; los nombres del
            archivo se comparan ya normalizados (sin tildes, mayusculas).
        contexto: etiqueta para mensajes de error (p. ej. "BASE_A").

    Returns:
        DataFrame solo con las columnas del contrato, renombradas.
    """
    mapa = {_norm(k): v for k, v in requeridas.items()}
    faltan = [k for k in mapa if k not in df.columns]
    if faltan:
        raise KeyError(
            f"[{contexto}] No se hallaron las columnas {faltan}.\n"
            f"  Columnas disponibles: {list(df.columns)}\n"
            "  Corrija los nombres en la CONFIGURACION (se comparan sin tildes ni mayusculas)."
        )
    out = df[list(mapa)].rename(columns=mapa).copy()
    for c in out.columns:
        out[c] = out[c].astype("string").fillna("").str.strip()
    return out


# ── Vistas derivadas (Excel si caben; CSV comprimido si exceden) ───────────
# La carpeta del estandar la escribe la libreria (un solo escritor:
# `escribir_resultado`). Lo que el notebook agrega son VISTAS derivadas de la
# correlativa, y van en `vistas/` dentro de esa carpeta: el manifiesto no las
# lista (no son parte del contrato) y `leer_resultado` las ignora. Las escribe
# `exportar_vistas` (la funcion libre del estandar, F2.11): un .xlsx por vista
# con las formulas neutralizadas, o un .csv.gz completo si no cabe en Excel.


@dataclass(frozen=True)
class EspecBase:
    """Describe UNA base de entrada y el mapeo de sus columnas al contrato.

    Attributes:
        nombre: etiqueta corta de la fuente (aparece en SRC), p. ej. "RUES".
        archivo: ruta local completa al .xlsx/.csv (p. ej. ``RUTA_DATOS / "base.xlsx"``).
        col_nit / col_nombre: como se llaman esas columnas EN SU archivo.
        col_ciudad: opcional; mejora precision si existe en ambas bases.
        hoja / fila_encabezado: ubicacion de los datos (solo Excel).
    """

    nombre: str
    archivo: str | Path
    col_nit: str
    col_nombre: str
    col_ciudad: str | None = None
    hoja: str | int = 0
    fila_encabezado: int = 0

    def __post_init__(self) -> None:
        if not self.nombre.strip():
            raise ValueError("EspecBase.nombre no puede estar vacio")
        if not str(self.archivo).lower().endswith((".csv", ".xlsx", ".xls", ".xlsm")):
            raise ValueError(f"[{self.nombre}] archivo debe ser .csv/.xlsx: {self.archivo}")

    def cargar(self) -> pd.DataFrame:
        """Lee la base y la deja en el contrato NIT/RAZON_SOCIAL[/CIUDAD]."""
        crudo = leer_base(self.archivo, self.hoja, self.fila_encabezado, (self.col_nit,))
        mapa = {self.col_nit: "NIT", self.col_nombre: "RAZON_SOCIAL"}
        if self.col_ciudad:
            mapa[self.col_ciudad] = "CIUDAD"
        df = exigir_columnas(crudo, mapa, self.nombre)
        print(f"[{self.nombre}] {len(df):,} registros listos")
        return df


def pares_cruzados(correlativa: pd.DataFrame, a: str, b: str) -> pd.DataFrame:
    """Pares A<->B que quedaron en el mismo ID_GRUPO (vectorizado, sin bucles)."""
    cols = ["ID_GRUPO", "NIT", "RAZON_SOCIAL", "SRC"]
    izq = correlativa.loc[correlativa["SRC"] == a, cols]
    der = correlativa.loc[correlativa["SRC"] == b, cols]
    pares = izq.merge(der, on="ID_GRUPO", suffixes=(f"_{a}", f"_{b}"))
    return pares.sort_values("ID_GRUPO", kind="stable").reset_index(drop=True)


def cruzar_dos_bases(
    base_a: EspecBase,
    base_b: EspecBase,
    confiable: str | None,
    usar_ciudad: bool,
    carpeta_salida: str | Path,
    perfil: str = "produccion_estandar",
) -> Path:
    """Flujo completo: cargar A y B -> link() -> carpeta del estandar -> vistas.

    Returns:
        La carpeta escrita: ``<carpeta_salida>/<AAAA-MM-DD_HHMM>_cruce_<A>_<B>/`` con
        ``golden.parquet``, ``correlativa.parquet``, ``excel/``, ``diccionario.csv``,
        ``revision.csv``, ``manifest.json`` y, en ``vistas/``, PARES_CRUZADOS y RESUMEN.
    """
    t0 = time.time()
    df_a, df_b = base_a.cargar(), base_b.cargar()
    con_ciudad = usar_ciudad and "CIUDAD" in df_a.columns and "CIUDAD" in df_b.columns

    r = link(
        df_a,
        df_b,
        nombre_a=base_a.nombre,
        nombre_b=base_b.nombre,
        trusted={confiable} if confiable else None,
        col_ciudad="CIUDAD" if con_ciudad else None,
        profile=perfil,
        work_dir=str(Path(carpeta_salida) / "trabajo"),
    )
    manifiesto = escribir_resultado(
        r, Path(carpeta_salida), nombre=f"cruce_{base_a.nombre}_{base_b.nombre}"
    )
    carpeta = manifiesto.carpeta
    print(f"Carpeta del estandar: {carpeta}")

    pares = pares_cruzados(r.correlativa, base_a.nombre, base_b.nombre)
    m = r.metricas
    resumen = pd.DataFrame(
        {
            "indicador": [
                f"registros_{base_a.nombre}",
                f"registros_{base_b.nombre}",
                "grupos_unicos",
                "grupos_cruzados_A_y_B",
                "pares_A_B",
                "perfil",
                "ciudad_usada",
                "duracion_seg",
                "carpeta",
            ],
            "valor": [
                m["n_registros_a"],
                m["n_registros_b"],
                m["n_grupos"],
                m["n_grupos_cruzados"],
                m["n_pares_a_b"],
                perfil,
                con_ciudad,
                round(time.time() - t0, 1),
                str(carpeta),
            ],
        }
    )
    for ruta in exportar_vistas(
        {"PARES_CRUZADOS": pares, "RESUMEN": resumen},
        carpeta / "vistas",
        f"cruce_{base_a.nombre}_{base_b.nombre}",
    ):
        print(f"  Vista escrita: {ruta}")
    print()
    print(r.resumen())
    del df_a, df_b, r
    gc.collect()
    return carpeta

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURAR Y EJECUTAR        (LA UNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
BASE_A = EspecBase(
    nombre="RUES",
    archivo=RUTA_DATOS / "base_rues.xlsx",
    hoja="Base",
    col_nit="NIT",
    col_nombre="RAZON SOCIAL",
    col_ciudad="CIUDAD",
)
BASE_B = EspecBase(
    nombre="CRM",
    archivo=RUTA_DATOS / "base_crm.xlsx",
    hoja=0,
    col_nit="NIT EMPRESA",
    col_nombre="NOMBRE",
    col_ciudad=None,
)
CONFIABLE = "RUES"  # fuente que NO se deduplica contra si misma (None si ninguna)
USAR_CIUDAD = True  # solo aplica si AMBAS bases declararon col_ciudad

CARPETA = cruzar_dos_bases(BASE_A, BASE_B, CONFIABLE, USAR_CIUDAD, RUTA_RESULTADOS)

# La carpeta se lee de vuelta con la libreria (verifica las huellas del manifiesto).
res = leer_resultado(CARPETA)
print(res.resumen())
pares_cruzados(res.correlativa, BASE_A.nombre, BASE_B.nombre).head(20)

### Como leer el resultado

`cruzar_dos_bases` deja UNA carpeta con la forma del estandar de salida (contrato 1.0):
`<RUTA_RESULTADOS>/<AAAA-MM-DD_HHMM>_cruce_<A>_<B>/`

- **golden.parquet** (y `excel/golden.xlsx`): un registro canonico por empresa (mejor NIT
  y nombre consolidados, `ID_ENTIDAD`).
- **correlativa.parquet** (y `excel/correlativa.xlsx`): todos los registros de ambas bases
  con su `ID_GRUPO`, `SRC`, `METODO_UNION` y `CONFIANZA`.
- **manifest.json**, **diccionario.csv**, **revision.csv**: trazabilidad, significado de
  cada columna y pares por decidir.
- **vistas/PARES_CRUZADOS**: cada fila es una empresa hallada en AMBAS bases, con el NIT y
  nombre tal como aparecen en cada una — el entregable tipico del cruce (se deriva de la
  correlativa: mismo `ID_GRUPO`, distinto `SRC`).
- **vistas/RESUMEN**: conteos de la corrida (`n_grupos_cruzados`, `n_pares_a_b`…).

`leer_resultado(carpeta)` devuelve el `ResultadoLinkage` (`res.golden`, `res.correlativa`,
`res.manifiesto`, `res.metricas`) verificando antes las huellas del manifiesto.

**`CONFIABLE` (trusted)**: declare aqui la base ya depurada (p. ej. RUES): el motor
no gastara esfuerzo deduplicandola contra si misma y el cruce sera mas rapido y limpio.
Si ninguna esta depurada, use `None`.